# Evaluación multisemilla por ganancia de múltiples experimentos z495

Esta notebook:

1. carga el dataset crudo y genera `clase_ternaria`;
2. lee el `BO_log.txt` de cada experimento indicado, generado por R o por Optuna/Python;
3. selecciona los mejores `n` conjuntos de hiperparámetros de cada experimento;
4. entrena cada conjunto con `m` semillas distintas sobre todos los datos de entrenamiento;
5. calcula y promedia la ganancia sobre el período futuro, como en Producción de z494.

No ejecuta una nueva optimización bayesiana.


## Librerías

Si falta algún paquete, instalarlo previamente en el entorno de R correspondiente.


In [1]:
library(data.table)
library(here)
library(primes)
library(lightgbm)

options(scipen = 999)


here() starts at /home/marco/code/DMEyF/dmeyf2026

Warning message:
“package ‘lightgbm’ was built under R version 4.3.3”


## Configuración

Cada elemento de `experimentos_origen` puede ser el nombre de una carpeta dentro de `DATA/EXP` o una ruta completa.


In [ ]:
root <- here()
dataset_folder <- file.path(root, "DATA", "DATASETS")
exp_folder <- file.path(root, "DATA", "EXP")

PARAM <- list()
# Ejemplo: c("HT4950-01", "HT4950-02")
PARAM$experimentos_origen <- c("PY495-01","PY495-02","PY4941", "PY495-01", "PY4942", "HT4941", "HT4942", "HT4940")
PARAM$archivo_resultados <- "BO_log.txt"

PARAM$semilla_primigenia <- 240707
PARAM$n_mejores <- 2L
PARAM$m_semillas <- 5L

PARAM$train_final <- c(202104)
PARAM$future <- c(202106)
PARAM$semilla_kaggle <- 314159
PARAM$cortes <- seq(4000, 19000, by = 500)
PARAM$trainingstrategy$undersampling <- 0.1


In [3]:
# Parámetros fijos de z495. Los presentes en BO_log.txt los pisan.
PARAM$lgbm$param_fijos <- list(
  boosting = "gbdt",
  objective = "binary",
  metric = "average_precision",
  feature_pre_filter = FALSE,
  first_metric_only = FALSE,
  boost_from_average = TRUE,
  force_row_wise = TRUE,
  deterministic = TRUE,
  verbosity = -100,

  seed = PARAM$semilla_primigenia,

  max_depth = -1L,
  min_gain_to_split = 0,
  lambda_l1 = 0.0,
  lambda_l2 = 0.0,
  max_bin = 31L,

  bagging_fraction = 1.0,
  pos_bagging_fraction = 1.0,
  neg_bagging_fraction = 1.0,
  is_unbalance = FALSE,
  scale_pos_weight = 1.0,

  drop_rate = 0.1,
  max_drop = 50L,
  skip_drop = 0.5,
  extra_trees = FALSE,

  early_stopping = 0L,
  min_data_in_leaf = 0L,
  feature_fraction = 0.50,
  learning_rate = 0.005,

  num_iterations = 20000L,
  num_leaves = 1024L,
  min_sum_hessian_in_leaf = 1e-6
)


## Dataset y clase ternaria

Se conserva la lógica de generación utilizada en z495.


In [4]:
dataset_url <- file.path(dataset_folder, "competencia_01_crudo.csv")

if (!file.exists(dataset_url)) {
  stop("No existe el dataset: ", dataset_url)
}

dataset <- fread(dataset_url)

dsimple <- dataset[, .(
  pos = .I,
  numero_de_cliente,
  periodo0 = as.integer(foto_mes / 100) * 12 + foto_mes %% 100
)]

setorder(dsimple, numero_de_cliente, periodo0)

periodo_ultimo <- dsimple[, max(periodo0)]
periodo_anteultimo <- periodo_ultimo - 1L

dsimple[, c("periodo1", "periodo2") :=
  shift(periodo0, n = 1:2, fill = NA, type = "lead"),
  by = numero_de_cliente
]

dsimple[periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA"]

dsimple[
  periodo0 < periodo_ultimo &
    (is.na(periodo1) | periodo0 + 1L < periodo1),
  clase_ternaria := "BAJA+1"
]

dsimple[
  periodo0 < periodo_anteultimo & periodo0 + 1L == periodo1 &
    (is.na(periodo2) | periodo0 + 2L < periodo2),
  clase_ternaria := "BAJA+2"
]

setorder(dsimple, pos)
dataset[, clase_ternaria := dsimple$clase_ternaria]

setorder(dataset, foto_mes, clase_ternaria, numero_de_cliente)
conteo_clases <- dataset[, .N, by = .(foto_mes, clase_ternaria)]


In [5]:
dataset[, clase01 := fifelse(
  clase_ternaria %in% c("BAJA+2", "BAJA+1"),
  1L,
  0L
)]

dataset_train <- copy(dataset[foto_mes %in% PARAM$train_final])
dfuture <- copy(dataset[foto_mes %in% PARAM$future])

campos_buenos <- setdiff(
  names(dataset_train),
  c(
    "clase_ternaria", "clase01", "fold", "azar", "training",
    "cprestamos_personales", "mprestamos_personales"
  )
)

dtrain_final <- lgb.Dataset(
  data = data.matrix(dataset_train[, ..campos_buenos]),
  label = dataset_train[, clase01]
)

conteo_train_final <- dataset_train[, .N, by = clase_ternaria]


## Lectura de resultados de R o Python

La función reconoce:

- `metrica` en el log de z495/R;
- `value` y columnas `params_*` en el log de Optuna/Python;
- `y` en otros logs de R compatibles.


In [6]:
resolver_carpeta_experimento <- function(experimento) {
  if (dir.exists(experimento)) {
    return(normalizePath(experimento, mustWork = TRUE))
  }

  candidato <- file.path(exp_folder, experimento)
  if (!dir.exists(candidato)) {
    stop("No existe la carpeta del experimento: ", candidato)
  }

  normalizePath(candidato, mustWork = TRUE)
}


leer_mejores_resultados <- function(carpeta, archivo, n_mejores) {
  ruta <- file.path(carpeta, archivo)
  if (!file.exists(ruta)) {
    stop("No existe el archivo de resultados: ", ruta)
  }

  tb <- fread(ruta)
  tb[, fila_origen := .I]

  if ("state" %in% names(tb)) {
    tb <- tb[state == "COMPLETE"]
  }

  # Optuna escribe los hiperparámetros como params_nombre.
  columnas_python <- grep("^params_", names(tb), value = TRUE)
  for (columna in columnas_python) {
    nombre_lgbm <- sub("^params_", "", columna)
    if (!nombre_lgbm %in% names(tb)) {
      setnames(tb, columna, nombre_lgbm)
    }
  }

  candidatas_metrica <- c("metrica", "value", "y")
  campo_metrica <- candidatas_metrica[candidatas_metrica %in% names(tb)][1]

  if (is.na(campo_metrica)) {
    stop("No se encontró una columna de métrica: metrica, value o y")
  }

  tb[, metrica_origen := as.numeric(get(campo_metrica))]
  tb <- tb[is.finite(metrica_origen)]

  if (nrow(tb) == 0L) {
    stop("El archivo no contiene evaluaciones completas con una métrica válida")
  }

  setorder(tb, -metrica_origen)
  tb <- head(tb, min(as.integer(n_mejores), nrow(tb)))
  tb[, ranking_origen := .I]

  if ("iter" %in% names(tb)) {
    tb[, id_origen := as.character(iter)]
  } else if ("number" %in% names(tb)) {
    tb[, id_origen := as.character(number)]
  } else {
    tb[, id_origen := as.character(fila_origen)]
  }

  tb[]
}


In [7]:
modelos_por_experimento <- lapply(
  PARAM$experimentos_origen,
  function(experimento) {
    carpeta <- resolver_carpeta_experimento(experimento)

    modelos <- leer_mejores_resultados(
      carpeta = carpeta,
      archivo = PARAM$archivo_resultados,
      n_mejores = PARAM$n_mejores
    )

    modelos[, experimento_origen := experimento]
    modelos
  }
)

mejores_modelos <- rbindlist(
  modelos_por_experimento,
  use.names = TRUE,
  fill = TRUE
)

mejores_modelos[, modelo_id := .I]

columnas_parametros <- intersect(
  names(PARAM$lgbm$param_fijos),
  names(mejores_modelos)
)

columnas_mostrar <- unique(c(
  "experimento_origen", "ranking_origen", "id_origen", "metrica_origen",
  columnas_parametros
))

# Descomentar si se quieren inspeccionar los modelos seleccionados.
# mejores_modelos[, ..columnas_mostrar]


## Semillas

Se generan semillas primas reproducibles y se utiliza `L'Ecuyer-CMRG`, siguiendo el manejo de semillas de las notebooks en R.


In [8]:
primos <- generate_primes(100000, 1000000)

if (PARAM$m_semillas > length(primos)) {
  stop("m_semillas supera la cantidad de números primos disponibles")
}

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$semillas <- sample(
  primos,
  size = PARAM$m_semillas,
  replace = FALSE
)

invisible(PARAM$semillas)


## Ganancia de Producción de z494

Se conserva la división fija `Public`/`Private` de z494. La semilla Kaggle es independiente de las semillas usadas para entrenar los modelos.


In [9]:
particionar <- function(
  data,
  division,
  agrupa = "",
  campo = "fold",
  start = 1L,
  seed = NA
) {
  if (!is.na(seed)) {
    set.seed(seed, kind = "L'Ecuyer-CMRG")
  }

  bloque <- unlist(mapply(
    function(x, y) rep(y, x),
    division,
    seq(from = start, length.out = length(division))
  ))

  data[, (campo) := sample(
    rep(bloque, ceiling(.N / length(bloque)))
  )[1:.N], by = agrupa]
}


realidad_inicializar <- function(pfuture, pparam) {
  drealidad <- pfuture[, .(
    numero_de_cliente,
    foto_mes,
    clase_ternaria
  )]

  particionar(
    data = drealidad,
    division = c(3L, 7L),
    agrupa = "clase_ternaria",
    seed = pparam$semilla_kaggle
  )

  drealidad
}


realidad_evaluar <- function(prealidad, pprediccion) {
  prealidad[
    pprediccion,
    on = c("numero_de_cliente", "foto_mes"),
    predicted := i.Predicted
  ]

  tbl <- prealidad[, .(
    qty = .N
  ), by = .(fold, predicted, clase_ternaria)]

  res <- list(
    public = tbl[
      fold == 1L & predicted == 1L,
      sum(qty * fifelse(clase_ternaria == "BAJA+2", 1072500, -27500))
    ] / 0.3,
    private = tbl[
      fold == 2L & predicted == 1L,
      sum(qty * fifelse(clase_ternaria == "BAJA+2", 1072500, -27500))
    ] / 0.7,
    total = tbl[
      predicted == 1L,
      sum(qty * fifelse(clase_ternaria == "BAJA+2", 1072500, -27500))
    ]
  )

  prealidad[, predicted := NULL]
  res
}


drealidad <- realidad_inicializar(dfuture, PARAM)


## Entrenamiento y evaluación de los mejores modelos

Cada fila del log pisa los parámetros fijos correspondientes. Cada semilla cambia el entrenamiento de LightGBM, pero no los hiperparámetros del experimento.


In [10]:
extraer_parametros_modelo <- function(fila_modelo, semilla) {
  disponibles <- intersect(
    names(PARAM$lgbm$param_fijos),
    names(fila_modelo)
  )

  parametros_log <- as.list(fila_modelo[, ..disponibles])
  parametros_log <- parametros_log[vapply(
    parametros_log,
    function(valor) length(valor) == 1L && !is.na(valor),
    logical(1)
  )]

  parametros <- modifyList(
    PARAM$lgbm$param_fijos,
    parametros_log
  )

  parametros$seed <- as.integer(semilla)

  parametros_enteros <- intersect(
    c(
      "num_iterations", "num_leaves", "max_depth", "max_bin",
      "max_drop", "early_stopping", "min_data_in_leaf"
    ),
    names(parametros)
  )

  parametros[parametros_enteros] <- lapply(
    parametros[parametros_enteros],
    as.integer
  )

  # Normalización de Producción de z494. En z495 permanece en cero.
  parametros$min_data_in_leaf <- round(
    parametros$min_data_in_leaf /
      PARAM$trainingstrategy$undersampling
  )

  parametros
}


entrenar_y_predecir <- function(fila_modelo, semilla) {
  parametros <- extraer_parametros_modelo(fila_modelo, semilla)

  modelo <- lgb.train(
    data = dtrain_final,
    param = parametros
  )

  prediccion <- predict(
    modelo,
    data.matrix(dfuture[, ..campos_buenos])
  )

  rm(modelo)

  prediccion
}


In [11]:
total_entrenamientos <- nrow(mejores_modelos) * length(PARAM$semillas)

resultados <- vector(
  mode = "list",
  length = total_entrenamientos * length(PARAM$cortes)
)

k <- 0L
entrenamientos_completos <- 0L
inicio_proceso <- Sys.time()

cat("Entrenamientos totales:", total_entrenamientos, "\n")
flush.console()

for (modelo_id in seq_len(nrow(mejores_modelos))) {
  cat("****MODELO:", modelo_id, "****** \n")
  flush.console()
  fila_modelo <- mejores_modelos[modelo_id]
  experimento <- fila_modelo$experimento_origen
  modelos_en_experimento <- mejores_modelos[
    experimento_origen == experimento,
    .N
  ]

  for (indice_semilla in seq_along(PARAM$semillas)) {
    semilla <- PARAM$semillas[indice_semilla]
    cat("****SEMILLA:", semilla, "****** \n")
    flush.console()
    inicio_entrenamiento <- Sys.time()

    prediccion <- entrenar_y_predecir(
      fila_modelo = fila_modelo,
      semilla = semilla
    )

    tb_prediccion <- dfuture[, .(
      numero_de_cliente,
      foto_mes,
      clase_ternaria
    )]

    tb_prediccion[, prob := prediccion]
    setorder(tb_prediccion, -prob)

    for (envios in PARAM$cortes) {
      k <- k + 1L

      tb_prediccion[, Predicted := 0L]
      tb_prediccion[seq_len(envios), Predicted := 1L]

      ganancia <- realidad_evaluar(
        prealidad = drealidad,
        pprediccion = tb_prediccion
      )

      resultados[[k]] <- data.table(
        modelo_id = modelo_id,
        experimento_origen = experimento,
        ranking_origen = fila_modelo$ranking_origen,
        id_origen = fila_modelo$id_origen,
        metrica_origen = fila_modelo$metrica_origen,
        semilla = as.integer(semilla),
        envios = as.integer(envios),
        ganancia_total = ganancia$total,
        ganancia_public_estimada = ganancia$public,
        ganancia_private_estimada = ganancia$private
      )
    }

    entrenamientos_completos <- entrenamientos_completos + 1L
    porcentaje <- 100 * entrenamientos_completos / total_entrenamientos
    minutos_transcurridos <- as.numeric(difftime(
      Sys.time(), inicio_proceso, units = "mins"
    ))
    minutos_ultimo <- as.numeric(difftime(
      Sys.time(), inicio_entrenamiento, units = "mins"
    ))
    minutos_restantes <-
      minutos_transcurridos / entrenamientos_completos *
      (total_entrenamientos - entrenamientos_completos)

    cat(sprintf(
      paste0(
        "[%d/%d - %.1f%%] Experimento=%s | modelo=%d/%d | ",
        "semilla=%d/%d | último=%.1f min | faltan~%.1f min\n"
      ),
      entrenamientos_completos,
      total_entrenamientos,
      porcentaje,
      experimento,
      fila_modelo$ranking_origen,
      modelos_en_experimento,
      indice_semilla,
      length(PARAM$semillas),
      minutos_ultimo,
      minutos_restantes
    ))
    flush.console()

    rm(prediccion, tb_prediccion)
    gc(full = TRUE, verbose = FALSE)
  }
}

resultados_detalle <- rbindlist(resultados)

# Descomentar para imprimir el resultado individual de cada semilla y corte.
# resultados_detalle[]


Entrenamientos totales: 80 
****MODELO: 1 ****** 
****SEMILLA: 150497 ****** 
[1/80 - 1.2%] Experimento=PY495-01 | modelo=1/4 | semilla=1/5 | último=1.8 min | faltan~143.0 min
****SEMILLA: 931363 ****** 
[2/80 - 2.5%] Experimento=PY495-01 | modelo=1/4 | semilla=2/5 | último=1.6 min | faltan~133.3 min
****SEMILLA: 240707 ****** 
[3/80 - 3.8%] Experimento=PY495-01 | modelo=1/4 | semilla=3/5 | último=1.4 min | faltan~123.9 min
****SEMILLA: 993683 ****** 
[4/80 - 5.0%] Experimento=PY495-01 | modelo=1/4 | semilla=4/5 | último=1.8 min | faltan~125.3 min
****SEMILLA: 121349 ****** 
[5/80 - 6.2%] Experimento=PY495-01 | modelo=1/4 | semilla=5/5 | último=1.6 min | faltan~123.3 min
****MODELO: 2 ****** 
****SEMILLA: 150497 ****** 
[6/80 - 7.5%] Experimento=PY495-01 | modelo=2/4 | semilla=1/5 | último=2.1 min | faltan~127.9 min
****SEMILLA: 931363 ****** 
[7/80 - 8.8%] Experimento=PY495-01 | modelo=2/4 | semilla=2/5 | último=1.9 min | faltan~128.3 min
****SEMILLA: 240707 ****** 
[8/80 - 10.0%] Exp

## Promedio de ganancias

`resumen_ganancias` muestra cada corte promediado sobre las `m` semillas. `mejor_corte_por_modelo` conserva el corte con mayor ganancia total promedio para cada conjunto de hiperparámetros.


In [12]:
resumen_metricas <- resultados_detalle[, .(
  ganancia_total_promedio = mean(ganancia_total),
  ganancia_total_sd = if (.N > 1L) sd(ganancia_total) else 0,
  ganancia_total_min = min(ganancia_total),
  ganancia_total_max = max(ganancia_total),
  ganancia_public_promedio = mean(ganancia_public_estimada),
  ganancia_private_promedio = mean(ganancia_private_estimada)
), by = .(modelo_id, envios)]

info_modelos <- mejores_modelos[, c(
  "experimento_origen",
  "ranking_origen",
  "id_origen",
  "metrica_origen",
  columnas_parametros
), with = FALSE]

info_modelos[, modelo_id := .I]

resumen_ganancias <- merge(
  info_modelos,
  resumen_metricas,
  by = "modelo_id"
)

setorder(resumen_ganancias, experimento_origen, ranking_origen, envios)

# Descomentar para imprimir el resumen de todos los cortes.
# resumen_ganancias[]


In [13]:
mejor_corte_por_modelo <- resumen_ganancias[
  order(modelo_id, -ganancia_total_promedio),
  .SD[1L],
  by = modelo_id
]

setorder(
  mejor_corte_por_modelo,
  experimento_origen,
  -ganancia_total_promedio
)

columnas_resumen <- intersect(c(
  "ranking_origen",
  "id_origen",
  "metrica_origen",
  "num_iterations",
  "num_leaves",
  "min_sum_hessian_in_leaf",
  "envios",
  "ganancia_total_promedio",
  "ganancia_total_sd",
  "ganancia_total_min",
  "ganancia_total_max",
  "ganancia_public_promedio",
  "ganancia_private_promedio"
), names(mejor_corte_por_modelo))

for (experimento in PARAM$experimentos_origen) {
  cat("\nExperimento:", experimento, "\n")
  print(
    mejor_corte_por_modelo[
      experimento_origen == experimento,
      ..columnas_resumen
    ]
  )
  flush.console()
}



Experimento: PY495-01 
   ranking_origen id_origen metrica_origen num_iterations num_leaves
1:              1         7      0.1186648           2645        488
2:              1         7      0.1186648           2645        488
3:              2        10      0.1180248           2938        834
4:              2        10      0.1180248           2938        834
   min_sum_hessian_in_leaf envios ganancia_total_promedio ganancia_total_sd
1:              0.09859814  13000               414700000           3810512
2:              0.09859814  13000               414700000           3810512
3:              0.04068127  12000               405680000           5944073
4:              0.04068127  12000               405680000           5944073
   ganancia_total_min ganancia_total_max ganancia_public_promedio
1:          411400000          420200000                442255000
2:          411400000          420200000                442255000
3:          399300000          412500000             

In [17]:
comparacion_modelos <- copy(mejor_corte_por_modelo)

setnames(
  comparacion_modelos,
  old = "envios",
  new = "mejor_corte"
)

comparacion_modelos[, cantidad_semillas := length(PARAM$semillas)]

columnas_comparacion <- intersect(c(
  "experimento_origen",
  "ranking_origen",
  "ganancia_total_promedio",
  "ganancia_total_sd",
  "ganancia_total_min",
  "ganancia_total_max",
  "ganancia_public_promedio",
  "ganancia_private_promedio",
  "id_origen",
  "metrica_origen",
  "num_iterations",
  "num_leaves",
  "min_sum_hessian_in_leaf",
  "cantidad_semillas",
  "mejor_corte"

), names(comparacion_modelos))

comparacion_modelos <- comparacion_modelos[, ..columnas_comparacion]

setorder(comparacion_modelos, -ganancia_total_promedio)
comparacion_modelos[, posicion_global := .I]
setcolorder(comparacion_modelos, "posicion_global")

comparacion_modelos[]

posicion_global,experimento_origen,ranking_origen,ganancia_total_promedio,ganancia_total_sd,ganancia_total_min,ganancia_total_max,ganancia_public_promedio,ganancia_private_promedio,id_origen,metrica_origen,num_iterations,num_leaves,min_sum_hessian_in_leaf,cantidad_semillas,mejor_corte
<int>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<int>,<int>,<dbl>,<int>,<int>
1,HT4942,1,449570000,3244534,444950000,452650000,473550000,439292857,96,0.9271928,2505,61,0.006727439,5,12500
2,HT4942,2,449020000,1967740,446600000,452100000,480865000,435372143,99,0.9271240,1081,97,0.008761439,5,14000
3,HT4940,1,446930000,4495220,440550000,450450000,467243333,438224286,29,0.9268956,516,132,NA,5,11500
4,PY495-02,1,442530000,5355091,436150000,447150000,473275000,429353571,5,0.1182831,2780,24,0.064010482,5,12500
5,PY4942,1,441100000,1347219,438900000,442200000,468086667,429534286,79,0.9256860,1101,10,0.004959059,5,12000
6,HT4941,1,438460000,4302674,432300000,442200000,443300000,436385714,58,0.9271038,1291,2039,NA,5,14000
7,PY4941,1,435270000,1434225,432850000,436150000,452320000,427962857,7,0.9243527,1306,691,0.006848539,5,14500
8,PY4942,2,434940000,4575260,430100000,441100000,461725000,423460714,60,0.9253601,1167,10,0.004139314,5,12000
9,PY4941,2,433840000,4832494,426800000,438900000,463576667,421095714,15,0.9228511,1176,644,0.006620613,5,11000


# Guardamos todo

In [14]:
marca_tiempo <- format(Sys.time(), "%Y%m%d_%H%M%S")
carpeta_salida <- file.path(
  exp_folder,
  paste0("benchmark_multisemilla_", marca_tiempo)
)

dir.create(carpeta_salida, recursive = TRUE)

semillas_usadas <- rbindlist(list(
  data.table(
    uso = "generacion_semillas",
    semilla = PARAM$semilla_primigenia
  ),
  data.table(
    uso = "particion_public_private",
    semilla = PARAM$semilla_kaggle
  ),
  data.table(
    uso = paste0("entrenamiento_", seq_along(PARAM$semillas)),
    semilla = PARAM$semillas
  )
))

fwrite(
  resultados_detalle,
  file.path(carpeta_salida, "resultados_detalle.tsv"),
  sep = "\t"
)

fwrite(
  resumen_ganancias,
  file.path(carpeta_salida, "resumen_por_corte.tsv"),
  sep = "\t"
)

fwrite(
  mejor_corte_por_modelo,
  file.path(carpeta_salida, "memejor_corte_por_modelo.tsv"),
  sep = "\t"
)

fwrite(
  semillas_usadas,
  file.path(carpeta_salida, "semillas_usadas.tsv"),
  sep = "\t"
)

saveRDS(
  PARAM,
  file.path(carpeta_salida, "PARAM.rds")
)

capture.output(
  sessionInfo(),
  file = file.path(carpeta_salida, "sessionInfo.txt")
)

cat("Resultados guardados en:", carpeta_salida, "\n")

Resultados guardados en: /home/marco/code/DMEyF/dmeyf2026/DATA/EXP/benchmark_multisemilla_20260910_100203 
